In [ ]:
%sql

-- VISTA LÓGICA DE NORMALIZACIÓN Y HOMOLOGACIÓN 
-- Se utilizan los atributo tags y titulo para inferir el tipo de producto 
-- Se agrega el atributo tienda
-- Se filtran ID nulos y/o corruptos

CREATE OR REPLACE VIEW vitivinicola_catalog.bronze.productos_limpios AS

SELECT 
    *,
    COALESCE(tipo, tipo_inferido) AS categoria
FROM (
select 
CASE 
  WHEN id RLIKE '^[0-9]+$' THEN CAST(id AS BIGINT)
  ELSE NULL
END AS id,
titulo,
CASE 
  WHEN bodega = 'Château Mouton-Rothschild' THEN 'Château Mouton Rothschild'
  WHEN bodega = 'Tenuta dell Ornellaia' THEN 'Tenuta dell''Ornellaia'
  WHEN bodega IN ('Paco y Lola', 'Paco & Lola') THEN 'Paco & Lola'
  WHEN bodega ='Bodega Norton' THEN 'Norton'
  when bodega= 'Tienda Susana Balbo' Then 'Susana Balbo'
  ELSE bodega
END AS bodega,
CASE 
  WHEN tipo LIKE 'Vino%' OR tipo LIKE 'vino%'OR tipo LIKE 'Wine'
    THEN 'Vino'
  WHEN tipo LIKE 'Espumante%' 
    THEN 'Espumante'
  WHEN tipo LIKE 'Grappa' OR tipo LIKE 'Pisco' OR tipo LIKE 'Vodka' OR tipo LIKE 'Spirits' 
    THEN 'Destilado'
  WHEN tipo LIKE 'Aceite%' OR tipo LIKE 'Mieles' 
    THEN 'Delicatessen'
  WHEN tipo LIKE 'Cajas Mix%' OR tipo LIKE 'Bebidas alcohólicas%'
    THEN 'Otros'
END AS tipo,
CASE 
  WHEN tags LIKE '%Vinos%' OR tags LIKE '%Tintos%'
    OR tags LIKE '%vinos%' OR tags LIKE '%tintos%'
    OR tags LIKE '%Blancos%' OR tags LIKE '%Rosado%' 
    OR tags LIKE '%Malbec%' OR tags LIKE '%Cabernet%' 
    OR tags LIKE '%Crios%' OR tags LIKE '%denuestracava%'
    THEN 'Vino'
  WHEN tags LIKE '%Espumosos%' 
    THEN 'Espumante'
  WHEN tags LIKE '%Vermouth%' OR tags LIKE '%Vermut%' OR tags LIKE '%Vermú%' 
    THEN 'Vermut'
  WHEN tags LIKE '%Destilados%' OR tags LIKE '%Whisky%' 
    OR tags LIKE '%Ron%' OR tags LIKE '%Mezcal%' 
    OR tags LIKE '%grappa%' OR tags LIKE '%Digestivo%'
    THEN 'Destilado'
  WHEN tags LIKE '%Catas%' OR tags LIKE '%Cursos%' THEN 'Sin clasificar'
  WHEN tags LIKE '%elicatessen%'  or  tags LIKE '%eilicatessen%' THEN 'Delicatessen'
  WHEN titulo LIKE '%ermouth%' OR titulo LIKE '%ermut%' 
  THEN 'Vermut'
  ELSE 'Sin clasificar'
END AS tipo_inferido,
tags,
CASE
  WHEN precio RLIKE '^[0-9]+(\.[0-9]+)?$' THEN CAST(precio AS DOUBLE)
  ELSE NULL
END as precio,
CASE
  WHEN precio_tachado RLIKE '^[0-9]+(\.[0-9]+)?$' THEN CAST(precio_tachado AS DOUBLE)
  ELSE NULL
END as precio_tachado,
TRY_CAST(stock AS BOOLEAN) AS stock,
publicado,
descripcion,
CASE 
    WHEN _source_file LIKE '%balbo%' THEN 'Susana Balbo'
    WHEN _source_file LIKE '%norton%' THEN 'Norton'
    WHEN _source_file LIKE '%ocio%' THEN 'Ocio Wine'
    WHEN _source_file LIKE '%miro%' THEN 'Exclusivas Miro'
    WHEN _source_file LIKE '%barrica%' THEN 'La Barrica'
END AS tienda,
pais,
moneda
from vitivinicola_catalog.bronze.productos
where pais IN ('Argentina','USA','Uruguay','España') 
AND precio RLIKE '^[0-9]+(\.[0-9]+)?$' 
AND id RLIKE '^[0-9]+$'
)

--De 3.199 registros originales, 1.894 resultan válidos tras filtrar IDs nulos y corruptos


In [ ]:
%sql

-- DISTRIBUCIÓN DE VALORES NULOS EN CATEGORIA
SELECT

    count(*) as Registros_Validos,
    count(*) - count(l.CATEGORIA) as Nulos
    from vitivinicola_catalog.bronze.productos_limpios l
order by 2 desc

-- Se recupero el 100% de registros con tipo Nulo mediante inferencia desde tags y titulo (1021 registros).

In [ ]:
%sql
-- DATASET GLOBAL DE PRODUCTOS VITIVINÍCOLAS 

CREATE OR REPLACE VIEW vitivinicola_catalog.bronze.productos_vino AS
SELECT *
FROM vitivinicola_catalog.bronze.productos_limpios
WHERE categoria IN ('Vino', 'Espumante', 'Vermut')


-- Contiene únicamente categorías relevantes para el análisis de mercado (Vino, Espumante y Vermut). 
-- Se excluyen 42 productos que corresponden a merchandising, experiencias y no vitivinícolas.


In [ ]:
%sql
-- ESTADISTICAS DE PRECIOS POR CATEGORÍA Y MONEDA

SELECT 
    categoria,
    moneda,
    COUNT(*) as registros,
    ROUND(MIN(precio), 2) as minimo,
    ROUND(MAX(precio), 2) as maximo,
    ROUND(AVG(precio), 2) as promedio,
    ROUND(PERCENTILE(precio, 0.01), 2) as p01,
    ROUND(PERCENTILE(precio, 0.25), 2) as p25,
    ROUND(PERCENTILE(precio, 0.50), 2) as mediana,
    ROUND(PERCENTILE(precio, 0.75), 2) as p75,
    ROUND(PERCENTILE(precio, 0.99), 2) as p99
FROM vitivinicola_catalog.bronze.productos_vino
WHERE moneda IN ('ARS', 'UYU')
GROUP BY moneda, categoria
order by moneda, categoria;

-- El análisis de precios y bodegas se limita a Argentina y Uruguay, evaluando cada mercado en su moneda local y 
-- sin conversión cambiaria. 

In [ ]:
%sql

-- VERIFICACION DE POSIBLES OUTLIERS 

SELECT titulo, bodega, precio, moneda, categoria
FROM vitivinicola_catalog.bronze.productos_vino
where moneda IN ('ARS', 'UYU')
AND precio > (
    SELECT PERCENTILE(precio, 0.99)
    FROM vitivinicola_catalog.bronze.productos_vino
    WHERE moneda IN ('ARS', 'UYU')
)
ORDER BY precio DESC

-- Se identificaron registros por encima del percentil 99. La revisión manual mostró que corresponden a 
-- presentaciones especiales y formatos no estándar. Al tratarse de productos reales, se conservaron dentro del 
-- análisis.


In [ ]:
%sql

-- DATASET ANALÍTICO REGIONAL DE PRODUCTOS VITIVINÍCOLAS 
-- Se crea segmentación de precios por cuartiles
 
CREATE OR REPLACE VIEW vitivinicola_catalog.bronze.precios_regional AS
WITH percentiles AS (
    SELECT 
        moneda,
        categoria,
        PERCENTILE(precio, 0.25) AS p25,
        PERCENTILE(precio, 0.75) AS p75
    FROM vitivinicola_catalog.bronze.productos_vino
    WHERE moneda IN ('ARS', 'UYU')
    GROUP BY moneda, categoria
)
SELECT 
    p.titulo,
    p.bodega,
    p.categoria,
    p.moneda,
    p.precio,
    p.pais,
    p.tags,
    CASE
        WHEN p.precio <= per.p25 THEN 'Económico'
        WHEN p.precio <= per.p75 THEN 'Medio'
        ELSE 'Premium'
    END AS segmento_precio,
    p.stock
FROM vitivinicola_catalog.bronze.productos_vino p
LEFT JOIN percentiles per
    ON p.moneda = per.moneda
    AND p.categoria = per.categoria
WHERE p.moneda IN ('ARS', 'UYU')




    
